# Stabilized Oseen flow and two-level adaptivity

The local equal-order formulation follows Araya, Cárcamo, Poza and Valentin (2021), DOI [10.1007/s10444-020-09833-8](https://doi.org/10.1007/s10444-020-09833-8). Variable convection requires its analytical divergence and a supplied magnitude bound. This notebook reads archived original pyMHM experiments; expensive acquisition is separate from routine CI.

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/8bce5fee1ca69e51bb364f57694a10b2b5e51cbe3565c765ec55e3c820716c9a/27_oseen_adaptivity-companion.zip"
COMPANION_SHA256 = "8bce5fee1ca69e51bb364f57694a10b2b5e51cbe3565c765ec55e3c820716c9a"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("flow/brinkman_oseen/27_oseen_adaptivity.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display



The equations are $-\nu\Delta u+\nabla u\,\beta+\gamma u+\nabla p=f$, $\nabla\cdot u=0$. The skew form includes $\gamma-\frac12\nabla\cdot\beta$. Stabilization uses opposite convection signs in trial and test residuals, positive grad-div and negative residual-product terms. Independent DOLFINx/UFL integration tests check the operators for local degrees one through three.

In [ ]:
data = root / "examples/results/oseen"
for path in sorted(data.glob("*.json")):
    record = json.loads(path.read_text())
    last = record["rows"][-1]
    print(path.stem, len(record["rows"]), {key: last[key] for key in ("velocity_l2", "pressure_l2", "effectivity")})


The estimator is $\eta_1+\eta_2$, with $\eta_2=2^{-2\ell}(\sum_K\eta_{2,K}^2)^{1/2}$ for Oseen. Effectivity is measured against the broken $V\times Q$ error, including gradients. Its constants are not one, and the experiments do not establish a universal viscosity-independent bound.

In [ ]:
for name in ("smooth-nu1-l1-uniform-history", "variable-nu1-l1-uniform-history"):
    display(Image(filename=str(root / f"docs/figures/oseen/{name}.png")))


Adaptation preserves the 16 macrotriangles, bisects selected trace segments and refines local meshes with an explicit matching-grid closure. The boundary and internal layers use the article's analytical fields with declared marking threshold and quadrature. Historical adaptive connectivity is not fully specified; these are not claims of identical table values. The internal-layer case has zero reaction and lies outside the strict positive-coercivity hypothesis.

In [ ]:
for name in ("boundary-nu0.01-l1-adaptive-fields", "boundary-nu0.01-l1-adaptive-history", "internal-nu0.001-l1-adaptive-fields", "internal-nu0.001-l1-adaptive-history"):
    display(Image(filename=str(root / f"docs/figures/oseen/{name}.png")))
